### Croma embeddings for each case of cloud filtering

This notebook:

Create the optical embeddings for the three different cases:
1. v1: QA60+SCL
2. v2: CS+ no dilation
3. v3: CS+ dilated

In [3]:
import os 
import glob
import numpy as np
import torch
import rasterio
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon
from pathlib import Path
from sklearn.decomposition import PCA
from torchgeo.models import croma_base, CROMABase_Weights

# ── Configuration ──
SCRIPT_DIR = os.path.dirname(os.path.abspath("__file__"))

if os.path.exists(os.path.expanduser("~/thesis_tiles_120px")):
    TILES_DIR = os.path.expanduser("~/thesis_tiles_120px")
    SHP_DIR = os.path.join(SCRIPT_DIR, "data_shp")
else:
    TILES_DIR = "/Users/angelicamariamorenorojas/Desktop/Master/thesis/tiles_120px"
    SHP_DIR = os.path.join(SCRIPT_DIR, "data_shp")

# CROMA config
PATCH_SIZE = 8       # 8x8 pixel patches
EMBED_DIM = 768      # ViT-B embedding dimension
TILE_SIZE = 120      # native CROMA tile size (multiple of 8, no cropping)
GRID_SIZE = TILE_SIZE // PATCH_SIZE  # 15x15 tokens

# S2 L2A bands in download order -> CROMA expects 12 bands (no B10)
CROMA_S2_BAND_INDICES = list(range(12))  # all 12 bands

# Device
device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available()
                      else "cpu")
print(f"Device: {device}")
print(f"Tiles dir: {TILES_DIR}")
print(f"CROMA: {TILE_SIZE}x{TILE_SIZE} -> {GRID_SIZE}x{GRID_SIZE} patch grid -> {EMBED_DIM}-dim tokens")

Device: cuda
Tiles dir: /share/home/e2406749/thesis_tiles_120px
CROMA: 120x120 -> 15x15 patch grid -> 768-dim tokens


## 1. Helper functions: loading, normalization, cropping

In [4]:
def normalize(x):
    """Per-channel robust normalization to [0, 1] (same as CROMA repository)."""
    x = x.float()
    imgs = []
    for ch in range(x.shape[1]):
        channel = x[:, ch, :, :]
        min_val = channel.mean() - 2 * channel.std()
        max_val = channel.mean() + 2 * channel.std()
        img = (channel - min_val) / (max_val - min_val + 1e-10)
        img = torch.clamp(img, 0, 1)
        imgs.append(img.unsqueeze(1))
    return torch.cat(imgs, dim=1)


def load_tile(tif_path, band_indices=None):
    """Load a 120x120 native-CROMA tile and return data, transform, crs."""
    with rasterio.open(tif_path) as src:
        data = src.read().astype(np.float32)
        transform = src.transform
        crs = src.crs
    if band_indices is not None:
        data = data[band_indices]
    return data, transform, crs

#Find the sample tile for a given fid 
def find_sample_tile(fid, product, window="evt"):
    pattern = os.path.join(TILES_DIR, product, f"fid_{fid}", window, "*", "tile_0.tif")
    tifs = sorted(glob.glob(pattern))
    return tifs[0] if tifs else None

#List all the fids with both S2 L2A and S1 GRD
def list_available_fids():
    s2_dir = os.path.join(TILES_DIR, "s2_l2a")
    s1_dir = os.path.join(TILES_DIR, "s1_grd")
    if not os.path.exists(s2_dir) or not os.path.exists(s1_dir):
        return []
    s2_fids = {d.replace("fid_", "") for d in os.listdir(s2_dir) if d.startswith("fid_")}
    s1_fids = {d.replace("fid_", "") for d in os.listdir(s1_dir) if d.startswith("fid_")}
    return sorted(s2_fids & s1_fids)

available_fids = list_available_fids()
print(f"FIDs with both S2 L2A and S1 GRD: {len(available_fids)}")
if available_fids:
    print(f"Examples: {available_fids[:10]}")
print(f"\nSingle forward pass: {TILE_SIZE}x{TILE_SIZE} -> {GRID_SIZE}x{GRID_SIZE} tokens (no sliding window needed)")

FIDs with both S2 L2A and S1 GRD: 41
Examples: ['101', '112', '113', '115', '116', '125', '13', '153', '156', '161']

Single forward pass: 120x120 -> 15x15 tokens (no sliding window needed)


In [5]:
def load_croma_model(modalities):
    model = croma_base(
        weights=CROMABase_Weights.CROMA_VIT,
        modalities=modalities,
        image_size=TILE_SIZE,
    )
    model = model.to(device)
    model.eval()
    return model

In [ ]:
scenarios = {
    'v1: QA60+SCL': pd.read_csv(os.path.join(DATA_DIR, 'v1_images_s2.csv')),
    'v2: CS+ no dilation': pd.read_csv(os.path.join(DATA_DIR, 'v3_images_s2.csv')),
    'v3: CS+ dilated': pd.read_csv(os.path.join(DATA_DIR, 'v3_images_s2.csv')),
}